# Chapter 5 — Deep Agents: Planning, Subagents & Filesystem Context (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare)

**Learning objectives**
- Explain the deep-agent pattern (planning + subagents + virtual filesystem)
- Decompose a research goal into subagent tasks
- Use a scratch filesystem for intermediate context
- Coordinate a supervisor over subagents

> Runtime: ~12 min (API)  
> Cost: paid LLM required  
> Data: synthetic research goal

> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

## Environment setup


### Secrets (Colab or local)


In [11]:
# @title Setting environmental variables
import os

# --- Dual-mode secrets: works in Google Colab AND locally (.env / environment) ---
try:
    from google.colab import userdata  # type: ignore

    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False

if not IN_COLAB:
    # Local run: load variables from a .env file if present (never commit .env!).
    try:
        from dotenv import load_dotenv

        load_dotenv()
    except Exception:
        pass


def get_secret(name, default=None):
    """Read a secret from Colab Secrets, else from local env/.env, else default."""
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)


# 👇 Choose your provider 👇
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"

if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LSH_OPENAI_API_KEY", "sk-...")
elif API_KEY_PROVIDER == "ANTHROPIC":
    os.environ["ANTHROPIC_API_KEY"] = get_secret(
        "LC4LSH_ANTHROPIC_API_KEY", "sk-ant-..."
    )
elif API_KEY_PROVIDER == "GEMINI":
    os.environ["GOOGLE_API_KEY"] = get_secret("LC4LSH_GOOGLE_API_KEY", "AIza...")
elif API_KEY_PROVIDER == "GROQ":
    os.environ["GROQ_API_KEY"] = get_secret("LC4LSH_GROQ_API_KEY", "gsk_...")

print(
    f"✅ API keys loaded for {API_KEY_PROVIDER} (source: {'Colab Secrets' if IN_COLAB else 'local env/.env'})"
)

# Hugging Face token (optional; needed for gated models)
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""

API keys loaded for OPENAI


### Install pinned dependencies


In [12]:
# @title Installing Python dependencies
%pip install -q "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "langchain-community==0.4.0" "langgraph>=0.2" "pydantic>=2.5" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [13]:
# @title Setting LangSmith variables
# ========================
# 👇 CONFIGURE HERE 👇
# ========================
LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY") or "lsv2_pt_..."
LANGSMITH_PROJECT = "lc4lsh-chapter5-deepagents"  # Traces appear under this name
REGION = "EU"  # "EU" or "US" - must match your account!
# ========================

if LANGSMITH_API_KEY and LANGSMITH_API_KEY != "lsv2_pt_...":
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ["LANGSMITH_ENDPOINT"] = (
        "https://eu.api.smith.langchain.com"
        if REGION == "EU"
        else "https://api.smith.langchain.com"
    )
    dashboard = (
        "https://eu.smith.langchain.com"
        if REGION == "EU"
        else "https://smith.langchain.com"
    )
    print(f"✅ LangSmith enabled!")
    print(f"   Region: {REGION} | Project: {LANGSMITH_PROJECT}")
    print(f"   Dashboard: {dashboard}")
else:
    print("⚠️ LangSmith disabled - paste your API key above to enable tracing")
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF")

✅ LangSmith enabled!
   Region: EU | Project: lc4lsh-chapter5-deepagents
   Dashboard: https://eu.smith.langchain.com


## What are "deep agents"?

**Deep agents** extend a simple tool-calling agent with three capabilities:

1. **Planning** — write and maintain an explicit plan (todo list)
2. **Subagents** — spawn focused sub-agents with their own context for subtasks
3. **Filesystem context** — a scratch workspace (real or virtual) to store intermediate results too large for the prompt

This pattern (popularized by `deepagents`) helps agents tackle long-horizon scientific tasks without overflowing context.


## 1. A virtual filesystem for context


In [14]:
class VFS:
    """A tiny in-memory filesystem the agent can read/write."""

    def __init__(self):
        self.files = {}

    def write(self, path, content):
        self.files[path] = content

    def read(self, path):
        return self.files.get(path, "")

    def ls(self):
        return list(self.files)


vfs = VFS()
vfs.write("/notes/background.md", "# Background Kinase Y is implicated in pathway Z.")
print("VFS files:", vfs.ls())

VFS files: ['/notes/background.md']


## 2. A planner that writes a todo plan


In [15]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)


def make_plan(goal):
    prompt = (
        f"Goal: {goal} Break it into 3-4 concrete subtasks as a numbered list. "
        f"Each subtask should be self-contained for a subagent."
    )
    plan = llm.invoke(prompt).content
    vfs.write("/plan.md", plan)
    return plan


plan = make_plan("Assess whether Compound X is a viable kinase-Y inhibitor")
print(plan)

To assess whether Compound X is a viable kinase-Y inhibitor, the following subtasks can be outlined:

1. **In Vitro Activity Assessment**: 
   - Conduct a series of biochemical assays to evaluate the inhibitory effect of Compound X on kinase-Y activity. This should include determining the IC50 value and assessing the specificity of Compound X against other kinases.

2. **Binding Affinity Analysis**: 
   - Perform surface plasmon resonance (SPR) or isothermal titration calorimetry (ITC) experiments to measure the binding affinity of Compound X to kinase-Y. This will help in understanding the strength and nature of the interaction between the compound and the target kinase.

3. **Cellular Efficacy Evaluation**: 
   - Test the effects of Compound X on cellular models that express kinase-Y. This should include assessing the impact on downstream signaling pathways, cell proliferation, and apoptosis to determine the biological relevance of the inhibition observed in vitro.

4. **Toxicity and

## 3. Subagents with isolated context


In [16]:
def subagent(name, task, context_paths=()):
    """Run a focused subagent; give it only the files it needs."""
    ctx = "".join(vfs.read(p) for p in context_paths if vfs.read(p))
    prompt = f"""You are subagent '{name}'.
    Context:
    {ctx}

    Task: {task}
    Produce a concise result."""
    out = llm.invoke(prompt).content
    vfs.write(f"/results/{name}.md", out)
    return out


r1 = subagent(
    "literature", "Summarize known kinase-Y inhibitors", ["/notes/background.md"]
)
r2 = subagent("docking", "Estimate binding feasibility of Compound X", [])
print("LITERATURE:", r1[:300])
print("DOCKING:", r2[:300])

LITERATURE: Kinase Y inhibitors are compounds that specifically target and inhibit the activity of kinase Y, which is involved in pathway Z. Some known inhibitors include:

1. **Inhibitor A** - A small molecule that selectively binds to the ATP-binding site of kinase Y, effectively blocking its activity.
2. **I
DOCKING: Binding feasibility of Compound X is estimated to be high based on its structural compatibility with the target binding site, favorable molecular interactions, and appropriate physicochemical properties. Further experimental validation is recommended to confirm these predictions.


## 4. Supervisor synthesizes subagent results


In [17]:
def supervisor(goal):
    results = "".join(
        f"### {p}{vfs.read(p)}" for p in vfs.ls() if p.startswith("/results/")
    )
    prompt = f"""Goal: {goal}
    Subagent results:
    {results}
    """ f"Synthesize a final verdict with confidence and caveats."
    verdict = llm.invoke(prompt).content
    vfs.write("/final.md", verdict)
    return verdict


verdict = supervisor("Assess whether Compound X is a viable kinase-Y inhibitor")
print(verdict)
print("VFS now:", vfs.ls())

Based on the gathered information, Compound X shows promise as a viable inhibitor of kinase Y. The assessment is supported by the following points:

1. **Literature Context**: There are several known inhibitors of kinase Y, each with distinct mechanisms of action, such as binding to the ATP-binding site or forming covalent bonds. This indicates that there is a precedent for developing effective inhibitors targeting this kinase, suggesting that Compound X could fit into this landscape.

2. **Docking Results**: The binding feasibility of Compound X is reported to be high, indicating that it has a strong potential to interact effectively with the target binding site of kinase Y. The favorable molecular interactions and appropriate physicochemical properties further bolster the likelihood of successful inhibition.

### Final Verdict:
**Confidence Level**: Moderate to High

**Caveats**:
- While the docking results are promising, they are computational predictions and require experimental va

## How this maps to `deepagents`

- **Planning** → the `/plan.md` todo written by `make_plan`
- **Subagents** → `subagent(...)` calls with isolated context windows
- **Filesystem** → the `VFS` scratch space decoupling intermediate results from the prompt

In the real `deepagents` library these are provided as middleware over a LangGraph agent, with a built-in `ls`/`read`/`write`/`edit` toolset and a todo-tracking tool.


## Limitations & safety notes

- This is a didactic re-implementation, not the `deepagents` package; install it for production features (middleware, real todo tools).
- Subagent isolation here is manual (we pass only selected files).
- The VFS is in-memory; use a real workspace dir for persistence.
- **Paid API required**.


---
### Further Reading

| Notebook | Relevance |
|----------|-----------|
| **Chapter 5 Building Personal Assistants LangGraph and Agents** | From single agents to deep-agent systems |
| **Chapter 7 Biological LangGraph SuperApp** | Biology-specific multi-agent SuperApp |
| **Chapter 9 Medical LangGraph HyperApp** | Clinical multi-agent HyperApp |


In [18]:
# Cleanup
import gc

for _v in ("llm", "model", "agent", "graph", "app", "workflow"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")

Cleanup complete.


## Exercises

<details><summary>Why use a filesystem for agent context?</summary>It offloads large intermediate results from the prompt, preventing context-window overflow on long tasks.</details>

<details><summary>Why isolate subagent context?</summary>Each subagent sees only what it needs, reducing cost and distraction and avoiding cross-talk.</details>

<details><summary>What does the supervisor do?</summary>It aggregates subagent outputs into a coherent final answer and maintains the overall plan.</details>

### Tasks
- **Task A** - Add an `edit(path, old, new)` VFS method and use it to refine `/plan.md`.
- **Task B** - Let the supervisor spawn a third subagent dynamically if a result is inconclusive.
- **Task C** - Persist the VFS to a real temp directory and reload it.
- **Task D** - Install `deepagents` and re-implement this workflow using its `create_deep_agent`.
